In [0]:
from pyspark.sql import functions as F

gold_risk_scored = spark.table("fraud_detection.gold.transactions_risk_scored")

daily_summary = (
    gold_risk_scored
    .withColumn("tx_date", F.to_date("transaction_date"))
    .groupBy("tx_date")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("is_fraudulent").alias("fraud_count"),
        F.round(F.sum("is_fraudulent") / F.count("*") * 100, 2).alias("fraud_rate_pct"),
        F.round(F.sum("transaction_amount_usd"), 2).alias("total_volume_usd")
    )
    .orderBy("tx_date")
)

daily_summary.write.mode("overwrite").saveAsTable("fraud_detection.gold.daily_fraud_summary")
display(spark.sql("SELECT COUNT(*) FROM fraud_detection.gold.daily_fraud_summary"))